# Dunnhumby seed 43 — 수정 원형 M4 가중 강도 0.25 단일 조건

기존 수정 원형 M4의 유효성 마스크와 가중식은 유지하고 **λ만 0.5 → 0.25**로 바꿉니다. M1과 λ=0.5 결과는 정확한 이전 보고서에서 재사용하며 새 학습은 M4 하나뿐입니다. M2/M5 학습은 이 노트북에 없습니다.

Dunnhumby 신규 상품 추천 개발구간(일 684–690), seed 43, binary graph, K=1 균등 미관측 음성, ID 64차원·2층·L2 1e-3. 최대 300 epoch, 25 epoch마다 개발평가, 전체 가격·구매금액 가중 적중값@10 최고점 선택(동률은 이른 점), 100 이후 4회 미개선 시 종료합니다. 기존 선택규칙을 바꾸지 않습니다.

판정: 선택 checkpoint에서 전체 가격·구매금액 가중 적중값@10과 가중 NDCG@10이 각각 M1보다 크고, Recall/NDCG @10·@20·@50 여섯 지표가 모두 M1의 99% 이상인지 각각 확인합니다. λ=0.5와의 차이, 모든 CLV구간, @20/@50 및 노출 지표도 원본에 저장합니다. 이 실행은 반복 노출된 개발분할의 단일 시드 탐색이며 유의성·일반화·CLV 귀속을 주장하지 않습니다. 최종 test/holdout은 만들지 않습니다.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
from google.colab import drive
ROOT = Path('/content/drive/MyDrive/논문/data')
REPORT = ROOT/'results_v3_dunnhumby_linear_nv_original_m4_validity_masked_seed43_v2/reports/result.json'
if REPORT.is_file():
    print('필요한 기존 결과가 보입니다. Drive를 재마운트하지 않습니다.')
else:
    if os.path.ismount('/content/drive'):
        raise RuntimeError('Drive는 연결됐지만 정확한 λ=0.5 보고서를 찾지 못했습니다. 계정과 REPORT 경로를 확인하세요. 학습은 시작되지 않았습니다.')
    try:
        drive.mount('/content/drive')
    except (ValueError, NotImplementedError) as exc:
        raise RuntimeError('Drive 연결 실패. 이 셀에서는 학습을 시작하지 않았습니다. Colab의 연결 계정을 확인하세요.') from exc
    if not REPORT.is_file():
        raise RuntimeError('Drive 연결 후에도 정확한 λ=0.5 보고서가 보이지 않습니다. 학습은 시작되지 않았습니다.')
SOURCE_COMMIT = 'REPLACE_AFTER_COMMIT'
REPO = Path('/content/clv-original-m4-lambda025-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
if 'lightgcn_clv_v3' in sys.modules:
    assert Path(sys.modules['lightgcn_clv_v3'].__file__).resolve().parent == REPO.resolve(), '다른 소스가 로드됐습니다. 런타임을 다시 시작하세요.'
os.chdir(REPO)
sys.path.insert(0, str(REPO))
import clv_original_m4_lambda025_screen as screen
import pandas as pd
OUT = ROOT/'results_v3_dunnhumby_original_m4_lambda025_seed43_v1'
print('코드 버전:', screen.VERSION, '| 새 모델:', screen.MODEL_ID)

## 1. 학습 전 출처·가중치 확인
정확한 이전 결과·M1 원본·기존 M4 checkpoint가 없거나 무효 입력에 추가 가중치가 있으면 학습 전에 중단합니다. 데이터 준비에는 시간이 걸릴 수 있지만 GPU 학습은 이 셀에서 시작하지 않습니다.

In [ ]:
cfg, prepared, audit = screen.prepare(REPORT, OUT)
assert cfg.positive_weight_lambda == 0.25 and cfg.m4_mode == 'original'
assert prepared['m4_diagnostics']['original_invalid_extra_absent']
print(json.dumps({'dataset': cfg.dataset, 'seed': cfg.seeds, 'new_arm': screen.MODEL_ID, 'lambda': cfg.positive_weight_lambda, 'max_epochs': cfg.epochs, 'selection': '기존과 동일: 전체 가중 적중값@10 최고점', 'final_test': False, 'holdout': False}, ensure_ascii=False, indent=2))
print(audit[audit.group.isin(['all', 'any_input_invalid'])].to_string(index=False))

## 2. M4 하나만 학습
중단되면 같은 노트북을 다시 실행하세요. 완료 epoch마다 optimizer와 난수상태가 저장되며 기존 결과 폴더의 동일 arm을 재개합니다. 별도 M1/M2/M5 학습은 하지 않습니다.

In [ ]:
import torch
assert torch.cuda.is_available(), '학습에는 GPU 런타임을 사용하세요.'
paths = screen.run(cfg, prepared)
print(json.dumps(paths, ensure_ascii=False, indent=2))

## 3. 전체 판독 및 원본 다운로드
아래 표는 요약이며 ZIP에는 전체·저/중/고CLV 절대지표, 모든 비교, 학습곡선, 판정 JSON과 무효 입력 감사가 들어 있습니다. 불리한 지표도 삭제하지 않습니다.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
absolute = pd.read_csv(paths['absolute'])
comparison = pd.read_csv(paths['comparison'])
report = json.loads(Path(paths['json']).read_text())
key_metrics = ['recall@10', 'ndcg@10', 'recall@20', 'ndcg@20', 'recall@50', 'ndcg@50', 'price_purchase_amount_weighted_hit@10', 'vndcg@10', 'price_purchase_amount_weighted_hit@20', 'vndcg@20', 'price_purchase_amount_weighted_hit@50', 'vndcg@50', 'coverage@10', 'coverage@20', 'coverage@50', 'user_value_tendency_recommended_price_alignment']
print(absolute[['model_id', 'selected_epoch', 'stopped_epoch'] + key_metrics].set_index('model_id').T.to_string())
print('사전 판정:', json.dumps(report['reading'], ensure_ascii=False, indent=2))
print(comparison[comparison.metric.isin(key_metrics)].to_string(index=False))
zip_path = Path('/content/original_m4_lambda025_seed43_results.zip')
with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as archive:
    for name, path in paths.items():
        archive.write(path, arcname=Path(path).name)
    for name in ('m4_validity_audit_lambda025.csv', 'm4_validity_audit_lambda025.json'):
        archive.write(OUT/name, arcname=name)
files.download(str(zip_path))